# 篇 3 · 读一份物理计划：先读哪三行，以及 AQE 的三段能力各管什么

> 要回答的坏结果：**"别人贴给你一份 plan，你从哪三行开始读，才知道该查并行度、该查键的分布、还是该查统计信息"**。
>
> 三篇分工（刻意不互相复述）：**篇 1** 回答"shuffle 后有个 task 跑了 40 分钟"（倾斜与热点 key 的**量法与解法**）；
> **篇 2** 回答"目录/文件爆炸、读它比算它还慢"（小文件与物理设计）；
> **本篇**回答"**这份计划本身在说什么**"——前两篇各取计划里的一列来判一件事，本篇教的是怎么读那一列，
> 以及 `AdaptiveSparkPlan` 为什么会让"你设的那个数"和"计划里最后那个数"不是同一个。
>
> 本篇把三件事量成了确定量（不是形容词）：
>
> 1. **shuffle 的宽度是被配置写的**：`shuffle.partitions=200` ⇒ 计划里就是 `hashpartitioning(k#…, 200)`，
>    换成 8 就是 8（②，marker `shuffle-partitions-is-a-plan-number`）。
> 2. **合并读侧的不是"AQE"那个大开关，是 `coalescePartitions` 那一个子开关**：三格对照实测
>    `200 / 1 / 200`（④b，marker `aqe-coalescing-is-one-switch`）。
> 3. **AQE 用的统计与 planner 用的估计不同源、不同时刻**：编译期那份挂在 logical plan 上
>    （`optimizedPlan().stats().sizeInBytes()`，实测 `dim` 600 B / `big` 3,200,000 B / join 输出 1,600,000,000 B），
>    运行时那份由 shuffle writer 落完之后汇总、挂在 `ShuffleQueryStage` 上（实测 `6.1 MiB` / `1200.0 B`），
>    **执行之前 formatted 文本里 `Statistics(` 出现 0 次**（④c 断的就是这一条）⇒
>    于是主阈值设成 `-1`（planner 谁都不广播）时，AQE 仍能把 join 当场改写成 `BroadcastHashJoin`（③ 与 ④c 各量一头）。

## 三条从前两篇继承的写法纪律（本篇一开始就按它写）

- **结论押确定量，耗时一律只打印。** 本篇所有 `assert` 判的都是"同一镜像每次跑都给同一个东西"的那一类：
  计划文本里的算子名与那个数字、执行后读侧的分区数、落盘文件数、配置出厂值。
  AQE 开/关的墙上时间（⑤ 打印三轮）一条都不 assert。
- **`explain()` 不能当判据。** 它 print 到 stdout 然后返回 `None`（篇 1 实测），
  `assert "X" in df.explain()` 是 `in None` ⇒ `TypeError`，红得毫无道理还会把人赶去改断言。
  本篇只在 Python 里用 `contextlib.redirect_stdout` 把它的文本**接进字符串**（接进来之后照样 print 给读者看）。
- **物理计划是 lazy val。** 换配置要看计划变化必须**每次新建 DataFrame**（篇 1 实测：复用同一个对象文本一字不变）。
  本篇每一格"换配置"的地方都是新建的。

## ⚠ 本篇顶回了简报里三处推测（都带实测依据，读者该知道哪句是量出来的）

1. **`executedPlan().toString()` 里根本没有 `AQEShuffleRead`。** 简报给的那条正则
   `AQEShuffleRead\s*(?:coalesced|partitioning …)` 在本镜像**永远匹配不到**：AQE 开着执行完，
   那一行打的是 `AdaptiveSparkPlan isFinalPlan=false` 底下**原样的** `Exchange hashpartitioning(k#22L, 200)`。
   `AQEShuffleRead` 只出现在 `explain("formatted")` 的输出里，而且**连分区数都不带**（实测原文见 ④a 打印的那几行）。
   ⇒ 本篇改成两条判据：**文本**用 formatted 判"有没有那一层"，**数字**用 `df.rdd.getNumPartitions()` 判"合成了几个"
   （实测它能反映 AQE 的合并结果：1 vs 200）。
2. **`ColumnarToRow` 不打断 `WholeStageCodegen`。** 简报的地图上写着"被 Exchange / AQEShuffleRead / ColumnarToRow 打断"，
   实测读 parquet + 聚合的计划里 `*(1)` 同时罩住 `ColumnarToRow`、`partial_count` 的 HashAggregate 和 `FileScan parquet`
   ——它三个在同一个 codegen stage 里（① 打印原文）。真正打断的只有 Exchange。
3. **"更少分区 ≠ 更快"这句在本机不是反例，只能当告诫。** 实测三轮 AQE 一侧都**更快**
   （⑤ 打印的三对样本：AQE 开一侧都在 0.05s 级、AQE 关一侧都在 0.41~0.47s 级，读侧 1 个分区 vs 200 个）
   ⇒ 本机既没证明也没证伪那句话：那 200 个桶里 156 个是空的（② 实测），又没有网络 shuffle，
   省下来的只是"调度 156 个空 task"的费用。所以它按纪律只进 markdown（⑤ 那一节），不进任何断言，也不进 marker 的名字。

## 本机能演示到哪一层（先说破，别拿本地的绿当分布式的证据）

- kernel 是 **`local[2]`**，不是 `local[*]`：一个 JVM、2 个线程、`--driver-memory 512m`，
  钉在 `docker/jupyter/kernels/arena-pyspark/kernel.json` ⇒ `defaultParallelism = 2`（② 那格的会话单元会打出来，
  并把它断成前提）。所以本篇凡出现"并行度"的数都要记得分母是 2。
- 没有节点、没有独立 executor ⇒ **`FetchFailedException`、executor OOM、broadcast 把某个 executor 撑爆、
  长尾 task 拖垮节点**这一族失效模式本机演示不了，本篇一律标「未验证」。
- `localShuffleReader` 在本机也量不出它的价值（它省的是"跨节点读同一个分区"，本机没有跨节点）。
- 数据与配置全部当场生成；临时目录走 `tempfile.mkdtemp(prefix="wi90-03-")`（容器内 `/tmp`），
  **不落进 `data/notebooks/`**（那是读者的工作区），收尾 `shutil.rmtree` 并断言它真的没了；
  notebook **不带 outputs**（结果属于执行的那一刻）。

内核请选 `PySpark (arena)`（`metadata.kernelspec.name` = `arena-pyspark`）——
交付前的闸门 `server/test/notebooks/tutorials.test.ts` 会**真执行这一篇**并按内核名判：
写成 `python3` 会跑在镜像自带的系统解释器上，那正是本项目划在外面的那条隔离红线。
闸门还按 `server/test/notebooks/tutorial-claims.ts` 登记的**四条**结论 marker 做**双向相等**核对：
少一条 = 结论被删/改跑/marker 漂格式；多一条 = 有结论没登记（它对闸门就隐形了）。
打 marker 的那个单元里还必须有语句级 `assert`（"把判断删了留着 print"会直接红）。
最后一个 code cell 是 `spark.stop()`：闸门跑完不留活内核。

*本文里出现的每个数字都是这一篇在本容器（Spark 3.5.5 / `local[2]` / 20 万行 / 50 个聚合键）实测出来的，
并注明在哪一格打印：出厂值由会话单元那张表打印，计划文本由 ①/④ 打印，读侧分区数由 ②④ 打印，
耗时由 ⑤ 打印。"从头执行一遍要多久"由**闸门**量：容器档那条用例三次报 **18.4 / 19.3 / 23.6 秒**
（量的是 `jupyter nbconvert --execute` 整篇跑完的墙钟，含 Spark 冷启动与 nbconvert 自身启动；目标 ≤90s、闸门预算 120s。
那 5 秒的差是同机负载造成的，不是本篇的形状变了 —— 这正是"秒数只打印、不进断言"的理由。
全篇最贵的一步是数据单元那一次 `big` 落 parquet 再读回来，实测 0.6~1.0 秒；其余每一步都在 0.05~1.1 秒之间）。
这些秒数**只打印、不断言**。跨版本行为（键名、默认值、AQE 规则名）一律标「未在本镜像逐版本核实，迁移以官方配置页为准」。*

In [ ]:
import contextlib
import io
import os
import re
import shutil
import tempfile
import time

from pyspark.sql import SparkSession, functions as f

spark = SparkSession.builder.appName("wi90-03-reading-the-plan-and-aqe").getOrCreate()
sc = spark.sparkContext

# 出厂值不许背，要当场读（篇 1 立的形状）。⚠ 在这台镜像上"读一个 conf"有三种结局，本篇后面要引用的
# 恰恰是"这个键有没有默认值"这一层，所以分开处理（三条都是本脚本实测，不是推测）：
#   ① 有出厂默认值 ⇒ conf.get 给字符串，SET -v 也列它；
#   ② 键存在但**从没设过、也没有默认值** ⇒ conf.get 给 Python None，SET -v 那一行的值是 <undefined>
#      （实测 spark.sql.adaptive.autoBroadcastJoinThreshold 就是这个形状 ⇒ 它的行为是"继承上面那个主键"，
#       这一条对 ④c 那格"是谁让 join 改成广播的"是决定性的）；
#   ③ 这个版本压根没有这个键 ⇒ conf.get **抛** SparkNoSuchElementException，**不是**返回 None
#      （实测拿 spark.sql.optimizer.statistics.columnSize 试的，见下面 NO_SUCH_KEY 那一行）。
# ⚠ 另外两个实测坑（篇 2 踩过，这里继续躲）：SET -v 不列某些存在的键；
#    conf.get(key, default) 的 default 会先按**该键的类型**转换 ⇒ 给整型键传字符串 default 直接抛
#    IllegalArgumentException 而不是返回 default。所以这里不传 default，改成接异常。
FACTORY_KEYS = [
    "spark.sql.shuffle.partitions",
    "spark.sql.adaptive.enabled",
    "spark.sql.adaptive.coalescePartitions.enabled",
    "spark.sql.adaptive.coalescePartitions.initialPartitionNum",
    "spark.sql.adaptive.coalescePartitions.minPartitionSize",
    "spark.sql.adaptive.coalescePartitions.parallelismFirst",
    "spark.sql.adaptive.advisoryPartitionSizeInBytes",
    "spark.sql.adaptive.autoBroadcastJoinThreshold",
    "spark.sql.autoBroadcastJoinThreshold",
    "spark.sql.adaptive.skewJoin.enabled",
    "spark.sql.adaptive.skewJoin.skewedPartitionFactor",
    "spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes",
    "spark.sql.adaptive.localShuffleReader.enabled",
    "spark.sql.adaptive.forceOptimizeSkewedJoin",
    "spark.sql.cbo.enabled",
]
NO_SUCH_KEY = "spark.sql.optimizer.statistics.columnSize"  # 第 ③ 种结局的标本


def conf_state(key):
    try:
        v = spark.conf.get(key)
    except Exception as e:  # noqa: BLE001
        return f"!!抛异常 {type(e).__name__}"
    return "(Python None ⇒ 键存在但没设过、也没有默认值)" if v is None else v


def simple(frame):
    '''物理计划文本。**取文本只有这一条路**（篇 1 实测过别的三种写法）：
    df.explain() 是 print 完返回 None；df._jdf.executedPlan() 与 df._jdf.physicalPlan() 在本镜像 Py4JError；
    DataFrame.explainString 在 3.5.5 不存在。
    '''
    return frame._jdf.queryExecution().simpleString()


def fmt(frame):
    '''explain("formatted") 的文本，接进字符串（本镜像**唯一**会打印 AQEShuffleRead 的模式，④a 实测）。
    接进来之后照样要 print 给读者看 —— 否则在 Jupyter 里跑这一篇的人什么也看不见。
    '''
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        frame.explain("formatted")
    return buf.getvalue()


setv = {str(r["key"]): str(r["value"]) for r in spark.sql("SET -v").collect()}
print("本篇会动到 / 要引用的键，出厂状态（本镜像实测；SET -v 那一列只是对照，缺项是真的缺）")
for k in FACTORY_KEYS:
    print("   ", k, "\n        conf.get:", conf_state(k), "| SET -v:", setv.get(k, "(SET -v 不列这个键)"))
print("   ", NO_SUCH_KEY, "（故意读的错键）\n        conf.get:", conf_state(NO_SUCH_KEY), "| SET -v:", setv.get(NO_SUCH_KEY, "(SET -v 不列这个键)"))
print("spark", spark.version, "| master", sc.master, "| defaultParallelism", sc.defaultParallelism)
n_adaptive = sum(1 for k in setv if k.startswith("spark.sql.adaptive"))
print("SET -v 里 spark.sql.adaptive.* 一共", n_adaptive, "个键 ⇒ ④ 那张『三段能力』表是从它们压出来的，上面只列本篇动过或要引用的")

# 两条前提断言：本篇后面每个数字都活在这两条之上。
# 红在这里说的是"这台镜像被换过"（kernel.json 的 local[2]、或 AQE 的出厂开关），不是本篇结论错了。
assert sc.defaultParallelism == 2, (
    f"defaultParallelism = {sc.defaultParallelism} ⇒ kernel 不再是 local[2]。"
    "本篇所有'并行度'意义上的数字（读侧合成几个分区、200 个桶落几个文件）都以 2 为分母，"
    "换了机器就要整篇重测，别改这条断言去迁就新数字"
)
assert spark.conf.get("spark.sql.adaptive.enabled") == "true", conf_state("spark.sql.adaptive.enabled")
print("⇒ 出厂 AQE 是**开**的；本篇 ② 那一格会显式关掉它，④ 再逐格打开（每格自己设、自己恢复）")

In [ ]:
N = 200_000  # 与篇 2 同量级。本篇判的全是"计划形状"，N 再大也不改变结论，只会改变耗时（⑤ 实测总账见导语）

big = spark.range(N).withColumn("k", f.col("id") % 50).withColumn("v", (f.col("id") % 100).cast("long"))
dim = spark.range(50).select(f.col("id").alias("k"), (f.col("id") * 7).cast("long").alias("dv"))

print("spark.range 出来的分区数 =", big.rdd.getNumPartitions(), "（= defaultParallelism；不是 200，也不是 2000）")
print("聚合键 k 的取值数 =", big.select("k").distinct().count(), "| 维表 dim 的行数 =", dim.count())
print("k 只有 50 个值 ⇒ 本篇**不**演示倾斜（那是篇 1）；这里要的是干净的计划形状")

# spark.range 没有 seed 参数（篇 1 实测 TypeError）；确定性来自表达式本身，换 seed 也换不出新形状。
assert big.rdd.getNumPartitions() == sc.defaultParallelism == 2, big.rdd.getNumPartitions()
assert big.select("k").distinct().count() == 50 and dim.count() == 50

SCRATCH = tempfile.mkdtemp(prefix="wi90-03-")
PARQUET = SCRATCH + "/big"
t = time.time()
big.write.mode("overwrite").parquet(PARQUET)
files = [x for x in os.listdir(PARQUET) if x.endswith(".parquet")]
print(f"为了 ① 那张地图里『FileScan 那一行长什么样』，把 big 落一次盘：{len(files)} 个文件 / {round(time.time() - t, 2)} s（目录在 {SCRATCH}，收尾会删并断言删掉了）")

scan = spark.read.parquet(PARQUET)
print("读回来的输入分区数 =", scan.rdd.getNumPartitions(), "⇒ 这一行是读侧的数，④a 会证明 AQE 管不着它（篇 2 同一件事）")
assert len(files) == 2, files
print("编译期估计（③ 与 ④c 都要用它，它是 planner 唯一能看到的大小）：")
for label, frame in (("big", big), ("dim", dim), ("big ⋈ dim", big.join(dim, "k", "inner"))):
    print("   ", label, "optimizedPlan().stats().sizeInBytes() =", frame._jdf.queryExecution().optimizedPlan().stats().sizeInBytes())
print("   ↑ 这个数是**编译期**挂在 logical plan 上的；AQE 决策用的不是它（④c 当场把两个数打在一起比）")

## ① 现象：一份物理计划的地图（先读哪三行）

下面那一格把三种形状的计划的**原文**打出来（②③④ 全都活在这三行上）。本镜像实测的聚合计划长这样：

```text
== Physical Plan ==
*(2) HashAggregate(keys=[k#22L], functions=[count(1)])
+- Exchange hashpartitioning(k#22L, 200), ENSURE_REQUIREMENTS, [plan_id=63]
   +- *(1) HashAggregate(keys=[k#22L], functions=[partial_count(1)])
      +- *(1) Project [(id#20L % 50) AS k#22L]
         +- *(1) Range (0, 200000, step=1, splits=2)
```

### 先读这三行

| 行 | 为什么是它 | 本镜像实测 | 读完这一步你就知道该查什么 |
| --- | --- | --- | --- |
| 1. `Range (0, 200000, step=1, splits=2)` / 换成读文件时是 `FileScan parquet … ReadSchema: struct<k:bigint>` | **源头开了几个分区**。`splits` 就是它，`FileScan` 那一行不带数字时要看它引用的 relation | `splits=2`（= `local[2]` 的 defaultParallelism）；落过盘的 2 个 parquet 读回来是 2 个输入分区 | 源头只有 2 ⇒ 后面所有 map 端 task 数都是 2；"聚合前太慢"先查这里，不是查 shuffle |
| 2. `Exchange hashpartitioning(k#22L, 200), ENSURE_REQUIREMENTS` | **那次 shuffle**：分区键、桶数、以及它是"谁要求"的 | `hashpartitioning(k#22L, 200)`；`shuffle.partitions=8` 时同一个位置变成 `(k#22L, 8)` | 那个 200 是**配置写的**，不是数据要的（② 判的就是这句）⇒ 键的基数小于它，注定有空桶 |
| 3. `*(1)` / `*(2)` 两个编号，以及中间夹着什么 | 编号是 **WholeStageCodegen 的 stage id**，**不是分区数**（这是最容易读错的一行） | 两个 `HashAggregate` 分别在 `*(1)` 和 `*(2)` 里，中间隔着那条 `Exchange` | codegen 被 `Exchange` 打断 ⇒ 一次 codegen 只覆盖一个 stage；看见 `*(N)` 别把它当并行度 |

### 三条要一起记住的"读法坑"（都是实测，不是常识背诵）

1. **`*(N)` 是 codegen stage 编号，不是分区数。** 计划文本里唯一能当"并行度"读的数字是
   `hashpartitioning(键, N)` 的第二个参数与 `Range(..., splits=N)`；`NumPartitions` 这件事在 AQE 之后还要再问一次（④a）。
2. **`ColumnarToRow` 不打断 codegen**（简报说它会，实测不对）：读 parquet + 聚合的原文是
   `*(1) HashAggregate(partial) +- *(1) ColumnarToRow +- FileScan parquet`，
   三层都挂在同一个 `*(1)` 上。打断 codegen 的实测只有 `Exchange`（以及 ④a 里 AQE 插在中间的那一层 shuffle 读）。
3. **`explain()` 返回 `None`。** 它是 print。任何 `assert "X" in df.explain()` 都是 `in None` ⇒ `TypeError`。
   本篇取文本走 `simple()`（= `simpleString()`），AQE 那一层走 `fmt()`（= 捕获 `explain("formatted")`），
   两个都在会话单元里定义好了。

### 为什么 AQE 的介入会让"你读的那份计划"和"真正跑的那份"不是一个东西

这是本篇的核心，也是 ④ 两格的落点：`spark.sql.adaptive.enabled=true` 时计划的根节点是
`AdaptiveSparkPlan isFinalPlan=false`，它带**两份**子树 —— 改写前的与改写后的。
本镜像实测（④a 会把原文打出来）：

| 什么时候取文本 | `explain("formatted")` 里有 `== Final Plan ==` 吗 | 有 `AQEShuffleRead` 吗 | `df.rdd.getNumPartitions()` |
| --- | --- | --- | --- |
| 新建 DataFrame、什么都没跑 | 没有 | **没有** | 1（⚠ 这个调用自己就把 shuffle 物化并 finalize 了，实测） |
| 只跑过 `df.count()` | 没有 | **没有** | — |
| 读过一次 `df.rdd`（= 逼出 Final Plan） | 有 | **有** | 1 |

⇒ 教学口径就一句话：**AQE 改写出来的算子只活在 Final Plan 里，而 Final Plan 要被"真的取一次分区"才逼得出来**；
`executedPlan().toString()` 那条路永远看不见它（本篇开头顶回的第 1 条）。

In [ ]:
# ① 只打印、不断言：这一格是给读者的地图，判据从 ② 开始。
spark.conf.set("spark.sql.adaptive.enabled", False)  # 先看"没有 AQE 插手时"计划长什么样（④ 再把它打开）
spark.conf.set("spark.sql.shuffle.partitions", 200)

print("=" * 78)
print("[A] 聚合（groupBy(k).count()），AQE 关，shuffle.partitions=200")
print("=" * 78)
print(simple(big.groupBy("k").count()))
print('''  读法：
   - 最后那行 `splits=2`      = 源头只有 2 个分区（local[2]），map 端就只有 2 个 task
   - 倒数第三行 `(k#22L, 200)` = 那次 shuffle 的桶数，由 spark.sql.shuffle.partitions 写的（② 判它）
   - `*(1)` / `*(2)`           = 两个 codegen stage，被那条 Exchange 切开（编号不是分区数）''')

print("=" * 78)
print("[B] 同一句聚合，但数据是**从 parquet 读回来的**（看 FileScan / ColumnarToRow 那两行）")
print("=" * 78)
print(simple(scan.groupBy("k").count()))
print('''  读法：
   - `FileScan parquet [k#67L] ... ReadSchema: struct<k:bigint>` 只列出**真正被用到的列** = 列裁剪发生在扫描层（`[k#67L]` 里的属性编号每次编译会变，别背它）
   - `*(1)` 同时罩住 HashAggregate(partial) / ColumnarToRow / FileScan ⇒ ColumnarToRow **没有**打断 codegen（顶回简报第 2 条）
   - `Batched: true` + `PushedFilters: []`：没下推任何过滤条件；哪天这里冒出内容，说明谓词下推生效了''')

print("=" * 78)
print("[C] join（big ⋈ dim），广播阈值关掉 ⇒ 两侧各自 shuffle")
print("=" * 78)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
print(simple(big.join(dim, "k", "inner")))
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10 * 1024 * 1024)
print('''  读法：两条 `Exchange hashpartitioning(..., 200)` + 两个 `Sort` + 一个 `SortMergeJoin`。
   "shuffle 后有个 task 跑 40 分钟"这种坏结果，先看这两条 Exchange 的**分区键是谁**、
   再看它是 join（本格）还是聚合（[A] 格）——分叉就在这里；键分布那条的查法是篇 1 的内容，本篇不重复。''')
print("恢复：shuffle.partitions =", spark.conf.get("spark.sql.shuffle.partitions"), "| autoBroadcastJoinThreshold =", spark.conf.get("spark.sql.autoBroadcastJoinThreshold"), "| adaptive =", spark.conf.get("spark.sql.adaptive.enabled"))

## ② `shuffle-partitions-is-a-plan-number`：计划里那个 200 是配置写的，不是数据要的

这一格判三件事（一句话的三个侧面，共用一个 marker）：

- **正面**：`shuffle.partitions=200` ⇒ 执行前 `df.rdd.getNumPartitions()` 给 200；换成 8 就给 8。
  ⇒ "那个数是被配置写进去的"这件事是**可复现的**，不是"200 是 Spark 的魔法默认值"这种传说。
- **文本对照**：同一格还要断到计划里 `hashpartitioning(k#…, 200)` / `(k#…, 8)` 的那个数字跟着变。
  只有数字断言不够：`df.rdd` 那条路哪天不经过 Exchange（例如 AQE 插了一层），"200"就可能来自别处。
- **空桶对照**：`k` 只有 50 个取值，却开 200 个桶 ⇒ 落盘只剩 44 个非空文件（实测，篇 2 那条"空分区不产文件"在这里只当尺子用）。
  ⇒ **"我把 shuffle.partitions 调大就更有并行度"是假的**：调大的那 150+ 个桶里根本没有数据可分，
  你多付出的是 200 个 task 的调度与 200 次 fetch 元数据。

⚠ 前提：这一格必须**关掉 AQE** 来量"计划里的数"。开着 AQE 时 `df.rdd` 那个调用本身就会把 shuffle 物化、
并按运行时统计重定分区数（④a 实测给 1），那就不是"配置写的那个数"了。
所以这里 AQE 关，④ 再把它开 —— 两格用的是同一个表达式，只差配置，读者能逐字对照。

### 闸门怎么证明这些 assert 不是装饰（四条破坏性验证，全部在容器档实测红过）

不是"写完就信"：每个坏法都故意改坏跑过一次（`content/` 是 bind mount ⇒ 改完容器立刻看到那份），
四条都看到 `GATE_EXIT=1`，做完再还原（`md5sum` 与变异前一致）并全量重跑一次确认全绿。
红话里的 cell 号是 notebook 单元的序号（从 0 数）：

| 故意改坏的东西 | 闸门红出来的话（原文摘要） |
| --- | --- |
| ③ 把两档阈值对调（`plan_off = join_plan_at(4096)`、`plan_mid = join_plan_at(-1)`） | `03-reading-the-plan-and-aqe.ipynb 有 cell 抛异常 … cell 8: AssertionError: expected [ 'cell 8: AssertionError' ] to deeply equal []` —— 只有那一格红（cell 8 就是 ③） |
| ④a 删掉"逼出 Final Plan"那两步（`_ = agg.count()` 与 `parts_actual = agg.rdd.getNumPartitions()`） | `cell 10: AssertionError`（红在 `assert "AQEShuffleRead" in txt_final`，正是简报要的"认不出 AQEShuffleRead"）+ `cell 13: NameError`（④c 拿不到 ④a 传下去的 `SHUFFLE_BYTES`，**连带** —— 与篇 2 那张表同一形状） |
| ② 把 `PLANNED = (200, 8)` 改成 `(8, 8)` | `cell 6: AssertionError` —— 红在"两档都跑过"那条覆盖面断言，**不是**红在 `KeyError`。这一条是实测之后改的：第一版把 `assert obs[200] == …` 写在前面，少一头时先炸 `KeyError`，红话说的是"作者写字典写错了"而不是"这一篇少了一档" |
| ④b 删掉 `coalescePartitions.enabled` 那一行（= 让它保持默认 true） | `cell 11: AssertionError`，断言消息里带着三格现在的数 ⇒ 现在它是 `(200, 1, 1)`，红话直接说"只关子开关那一格也被合并了"。**这一条同时证明这一篇判的是那个子开关，而不是 AQE 的大开关** |

四条做完把文件还原（`md5sum` 回到变异前那份），再跑一次**全量**（三篇都跑）：`12 passed / 0 skipped`，
那一次三篇各是 28.8 / 62.7 / 23.6 秒（前两次全量是 25.7 / 55.2 / 18.4 与 26.5 / 56.4 / 19.3 秒）。
外加一条不需要容器的：打 marker 的那个单元里**删掉 assert、留着 marker 那行 print** ⇒ 宿主档 45 秒就红在
「每条已登记 marker 所在的那个 code cell 里必须至少有一条 assert」（`tutorials.test.ts` 常驻那组，判的是盘上字节）。
所以那些 `WI90[...] OK` 不是装饰：别删、别改格式、也别把判断删掉只留打印。

In [ ]:
# marker shuffle-partitions-is-a-plan-number gate 的是上面那**一句结论**：
# "shuffle 之后的分区数是配置写的，不是数据要的"。这个 cell 里的四条 assert 各自是它的哪一面：
#   ① 前提（AQE 必须关着）——它不成立时下面两条量的就不是"配置写的那个数"；
#   ② 正面：每档里 getNumPartitions() 与配置值相等、且计划文本里 hashpartitioning(键, N) 的 N 跟着配置动
#      （这两条在循环里逐档断，红话说的是"哪一档不对"）；
#   ③ 覆盖面：两档都真的跑过（sorted(obs) 那条，专门堵"把 (200, 8) 改成单档"这种减法）；
#   ④ 空桶对照：桶数 > 键基数时多出来的桶是空的（落盘文件数只有 44）。
spark.conf.set("spark.sql.adaptive.enabled", False)
assert spark.conf.get("spark.sql.adaptive.enabled") == "false", (
    f"adaptive = {spark.conf.get('spark.sql.adaptive.enabled')} —— ② 的度量前提是**关掉 AQE**。"
    "红在这里说的是上面那行配置被删了/被改回去了，不是结论消失了："
    "开着 AQE 时 df.rdd 会按运行时统计重定分区数（④a 实测给 1），量的就不是'配置写的那个数'了。别改下面的断言，把配置加回去"
)

PLANNED = (200, 8)   # 破坏性验证 ③ 改的就是这个元组：改成 (8, 8) 必须红，而且红在断言上、不是红在 KeyError 上
obs = {}
for sp in PLANNED:
    spark.conf.set("spark.sql.shuffle.partitions", sp)
    agg = big.groupBy("k").count()          # ⚠ 每次新建：物理计划是 lazy val，复用对象改了配置文本也不变
    txt = simple(agg)
    m = re.search(r"hashpartitioning\([^,]+,\s*(\d+)\)", txt)
    obs[sp] = (agg.rdd.getNumPartitions(), m.group(1) if m else None)
    print(f"shuffle.partitions = {sp:>3} → df.rdd.getNumPartitions() = {obs[sp][0]:>3} | 计划里 hashpartitioning(k#…, {obs[sp][1]})")
    assert obs[sp] == (sp, str(sp)), (sp, obs[sp])

# "两头都真的跑过"这件事本身也要断住（放在按键取值之前，否则少一头时先炸 KeyError，
# 红话说的是"作者写错了字典"而不是"这一篇少量了一档"—— 那种红会把人赶去改测试而不是改教程）。
assert sorted(obs) == sorted(PLANNED) == [8, 200], (sorted(obs), sorted(PLANNED))

# 空桶对照：50 个键开 200 个桶，多出来的桶天生是空的。
spark.conf.set("spark.sql.shuffle.partitions", 200)  # ⚠ 上面那个循环把配置留在了 8，这里必须显式回 200
EMPTY = SCRATCH + "/emptybuckets"
big.groupBy("k").count().write.mode("overwrite").parquet(EMPTY)
n_data = len([x for x in os.listdir(EMPTY) if x.endswith(".parquet")])
shutil.rmtree(EMPTY, ignore_errors=True)
print(f"同样这一句聚合 @200 桶，落盘非空文件只有 {n_data} 个 ⇒ {200 - n_data} 个桶是空的（k 只有 50 个值）")
assert n_data <= 50 < 200, n_data
print(f"⇒ 『把 shuffle.partitions 调大就更有并行度』在这份数据上是假的：并行度上限是键的基数，不是那个配置值")
print("WI90[03-reading-the-plan-and-aqe][shuffle-partitions-is-a-plan-number] OK")

spark.conf.set("spark.sql.shuffle.partitions", 200)

## ③ `broadcast-threshold-changes-join`：阈值改的不只是算子名，是**两条 Exchange 的有无**——而它比的那个数当场量给你看

篇 1 的 ③B 已经从"绕开 join 倾斜"的角度量过一次算子名。本篇不重复那个动机，只做两件篇 1 没做的事：

1. **把"广播省掉的是两次全量 shuffle"数出来**：计划里 `Exchange hashpartitioning(` 出现的次数从 2 变 0。
2. **把"阈值比的是哪一个数"钉死**：用三档阈值把编译期估计**夹在中间**。本镜像实测
   `dim` 的 `optimizedPlan().stats().sizeInBytes()` = **600 字节**、`big` = **3,200,000 字节**
   （会话单元与下面那一格各打印一次），于是：

| `autoBroadcastJoinThreshold` | 与两份编译期估计的关系 | 实测计划 | `hashpartitioning` 的 Exchange 条数 |
| --- | --- | --- | --- |
| `-1` | 显式关掉 | `SortMergeJoin` | 2（两侧各自全量 shuffle） |
| `512` | 比 `dim`(600 B) 还小 ⇒ 谁都别广播 | `SortMergeJoin` | 2 |
| `4096` | `600 < 4096 < 3,200,000` ⇒ 只广播小的一侧 | `BroadcastHashJoin … BuildRight` | **0** |

⇒ "10MB 以下自动广播"这句话里**主语是那个估计值**：既不是文件的真实字节，也不是运行时统计
（④c 会看见 AQE 用的正是后者）。把 `512` 与 `4096` 两档摆在一起而不是只演示"开/关"，
判的就是"跟着这个数走"这一句 —— 只有 `-1 / 10MB` 两档的话，阈值改成 20MB 也照样绿，那什么也没判住。

⚠ 两个必须写下来的坑：

- **每次换阈值都要新建 DataFrame**（篇 1 实测的 lazy val 坑：复用对象的话文本一字不变，
  会把"阈值改变了计划"判成假的）。下面那个 `join_plan_at()` 就是为这件事存在的。
- 题库 `bd-pyspark-0002` 的题面写"`autoBroadcastJoinThreshold` 默认 8 MB"，本镜像实测出厂是 **10 MB**
  （`10485760b`，会话单元打印的那一行）。**哪个版本起是 8MB 本篇没有逐版本核实**，迁移以官方配置页为准；
  别把这里读到的 10MB 当跨版本常数背。

In [ ]:
# marker broadcast-threshold-changes-join gate 的是**一句**结论：
# "广播阈值改的是这一格 join 的算子名与 Exchange 的有无，而决定它的是编译期规模估计那个数"。
# 这个 cell 的 assert 全是那句话的侧面：① 三档各自的算子名（正面）；② 反向两头
# （关掉/低于估计时不许出现 Broadcast、放开时不许残留 SortMergeJoin —— 篇 1 立的形状）；
# ③ Exchange 计数 2 / 2 / 0（本篇新加的那一条）；④ 512 与 4096 把两份估计夹住
# ——它不成立时"是那个估计值决定了广播"这句话就没依据，所以它同属这一条结论。
spark.conf.set("spark.sql.adaptive.enabled", False)   # ③ 只看 planner 自己的选择，④c 再看 AQE 改不改
spark.conf.set("spark.sql.shuffle.partitions", 200)
THRESHOLD_ON = 10 * 1024 * 1024


def join_plan_at(threshold):
    '''按给定阈值取一次 join 的物理计划。⚠ 每次新建 DataFrame：计划是 lazy val，复用对象不会重新选策略。'''
    spark.conf.set("spark.sql.autoBroadcastJoinThreshold", threshold)
    return simple(big.join(dim, "k", "inner"))


def join_lines(txt):
    return "\n".join(ln.strip() for ln in txt.splitlines() if "Join" in ln or "Broadcast" in ln)


est = {name: fr._jdf.queryExecution().optimizedPlan().stats().sizeInBytes() for name, fr in (("big", big), ("dim", dim))}
plan_off = join_plan_at(-1)      # 显式关掉
plan_tiny = join_plan_at(512)    # 比 dim 的估计还小 ⇒ 谁都别广播
plan_mid = join_plan_at(4096)    # 夹在两份估计之间 ⇒ 只广播小的一侧
n_ex = {"off(-1)": len(re.findall(r"Exchange hashpartitioning\(", plan_off)),
        "512": len(re.findall(r"Exchange hashpartitioning\(", plan_tiny)),
        "4096": len(re.findall(r"Exchange hashpartitioning\(", plan_mid))}
for label, txt in (("阈值关掉 (-1)", plan_off), ("阈值 512（比 dim 的估计还小）", plan_tiny), ("阈值 4096（夹在 600 与 3,200,000 之间）", plan_mid)):
    print(f"--- {label} ---")
    print(join_lines(txt))
print("编译期规模估计（阈值比的就是这个数）:", est, "| 三档各自的 hashpartitioning Exchange 条数:", n_ex)

assert "SortMergeJoin" in plan_off and "Broadcast" not in plan_off, plan_off
assert "SortMergeJoin" in plan_tiny and "Broadcast" not in plan_tiny, plan_tiny
assert "BroadcastHashJoin" in plan_mid and "SortMergeJoin" not in plan_mid, plan_mid
assert "BuildRight" in plan_mid, plan_mid          # 广播的是 dim 那一侧（600 字节那一个）
assert n_ex == {"off(-1)": 2, "512": 2, "4096": 0}, n_ex
assert 512 < est["dim"] < 4096 < est["big"], est
print(f"⇒ 数得出来的是『两条 hashpartitioning 的 Exchange 消失、换成一个 BroadcastExchange』（{n_ex['512']} → {n_ex['4096']}）；")
print(f"   而决定它的是 {est['dim']} 与 {est['big']} 这两个**编译期**估计，不是文件真实字节、也不是运行时统计")
print("WI90[03-reading-the-plan-and-aqe][broadcast-threshold-changes-join] OK")

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", THRESHOLD_ON)   # 恢复出厂（会话单元打印的那个 10MB）

## ④ AQE 三段能力各管什么：谁能当场量、谁在本机根本触发不了

AQE 不是"一个开关"，是**一条执行路径 + 挂在它上面的几条改写规则**。本镜像 `SET -v` 里 `spark.sql.adaptive.*`
一共 17 个键（会话单元会打印这个数，本篇只引用其中 15 个 + 一个故意读错的键），本篇把它们压成三段：

| 段 | 干的事 | 开关 | 本篇能不能当场量 |
| --- | --- | --- | --- |
| 合并读侧分区 | shuffle 物化后按真实字节/行数把小分区合成大分区 | `spark.sql.adaptive.coalescePartitions.enabled` | ✅ ④a/④b：200 → 1，关掉子开关 → 200 |
| 动态 join 策略 | 子 stage 物化后重挑 join 策略/构建侧（含按真实大小决定广播） | `spark.sql.adaptive.autoBroadcastJoinThreshold`（出厂 `<undefined>` ⇒ 继承主键） | ✅ ④c：初始 `SortMergeJoin` → Final `BroadcastHashJoin` |
| 倾斜 join 改写 | 把超过阈值的倾斜分区劈开、另一侧复制补齐 | `spark.sql.adaptive.skewJoin.enabled`（出厂 `true`，阈值实测 `256MB` + `factor 5.0`） | ❌ **本机不可能触发**：④a 量到的 shuffle 总量是 `3.1 KiB`（= 3,174 字节，实测并打印），比 256MB 小 5 个数量级 ⇒ 数学上没有任何一个分区能过阈值。只能给口径，见 ⑥ 的"没验的东西" |

### ④a 那两格为什么这么设计（先看实测，再写断言）

`AQEShuffleRead` 这个算子是 AQE 合并读侧留下的**唯一物理痕迹**。简报猜它是从
`df._jdf.queryExecution().executedPlan().toString()` 里正则抓出来的，本镜像实测**抓不到**：
那条文本在 AQE 开着、执行完之后打的仍然是 `AdaptiveSparkPlan isFinalPlan=false` 底下原样的
`Exchange hashpartitioning(k#22L, 200)`。真正有它的是 `explain("formatted")`，而且**不写分区数**：

```text
   +- == Final Plan ==
      * HashAggregate (7)
      +- AQEShuffleRead (6)
         +- ShuffleQueryStage (5), Statistics(sizeInBytes=3.1 KiB, rowCount=100)
```

所以本篇的两条判据各用一种观测：

- **④a `aqe-coalesces-shuffle-read`** 判"读侧确实被合并了，而且合并的依据是 AQE 自己量到的运行时统计"：
  数字走 `df.rdd.getNumPartitions()`（实测 1），痕迹走 formatted 文本里的 `AQEShuffleRead` / `ShuffleQueryStage`
  那两行 + 那句 `Statistics(sizeInBytes=…)`（它同时是 ④c 与 skewJoin 那一段的证据）。
- **④b `aqe-coalescing-is-one-switch`** 判"这是那**一个**子开关干的，不是 AQE 那个大开关"：三格对照
  `(大开关关, 子开关开) → 200`、`(开, 开) → 1`、`(开, 关) → 200`，
  并且第三格里 `ShuffleQueryStage` 与 `Statistics(…)` 两行**都还在**（实测）——
  合的是 AQEShuffleRead 那一层，统计采集那一层没被关掉。这一句就是面试追问要落的机制层。

⚠ 顺序判据（实测，不是推测）：**只 `count()` 不算数**。`df.count()` 执行的是"在它上面再加一层全局聚合"的另一棵树，
`df` 自己的 `AdaptiveSparkPlan` 还停在 `isFinalPlan=false`，formatted 里 `AQEShuffleRead` 一个字都没有；
要读过一次 `df.rdd`（= 逼它 finalize）才看得见。④a 把这两种状态都取一遍文本并**断言它们不同**，
所以"把物化那一步删掉 ⇒ 红在认不出 AQEShuffleRead"这条破坏性验证是真有牙的
（简报写的"只删 `_ = agg.count()`"在本镜像删不红 —— 因为 `df.rdd` 自己会物化，见 ④a 的打印）。

⚠ AQE 开着时 `spark.sql.shuffle.partitions` 那个数是**上限不是结果**：篇 1 实测的"200 个 shuffle 分区只落 2 个文件"、
篇 2 实测的"聚合那一路 199 → 1，而显式 `repartition(200)` 两档都落 200"都是这一条的下游后果。
④b 里被合并的那个 Exchange 来自聚合自己的 hash shuffle，**不是**显式 `repartition` —— 用显式 repartition 这一格就测不出合并了。

In [ ]:
# marker aqe-coalesces-shuffle-read gate 的是**一句**结论：
# "AQE 会把聚合那次 shuffle 的读侧分区合并掉，而它依据的是自己物化出来的运行时统计"。
# 这个 cell 的 assert 各自的面向：① 计划里那个"配置写的数"是 200（前提，②已经判过它的来源）；
# ② Final Plan 里有 AQEShuffleRead 那一层（痕迹）；③ 读侧分区数 1 < 200（数字）；
# ④ Statistics 那句能解析出字节数（= 它依据的是运行时统计，不是编译期估计）。
spark.conf.set("spark.sql.adaptive.enabled", True)
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", True)
spark.conf.set("spark.sql.shuffle.partitions", 200)

agg = big.groupBy("k").agg(f.sum("v").alias("s"), f.count(f.lit(1)).alias("n"))   # 新建（lazy val）
txt_initial = fmt(agg)
m_plan = re.search(r"hashpartitioning\([^,]+,\s*(\d+)\)", simple(agg))
assert m_plan, simple(agg)
parts_planned = int(m_plan.group(1))
print("执行前 formatted 文本里 AQEShuffleRead 出现次数 =", txt_initial.count("AQEShuffleRead"), "（实测 0：这时候只有 Initial Plan）")
assert "AQEShuffleRead" not in txt_initial, txt_initial.count("AQEShuffleRead")

t = time.time()
_ = agg.count()
print("count() 跑了", round(time.time() - t, 2), "s")
txt_after_count = fmt(agg)
print("只跑 count() 之后 AQEShuffleRead 出现次数 =", txt_after_count.count("AQEShuffleRead"), "| 有 == Final Plan == 吗：", "== Final Plan ==" in txt_after_count)
assert "AQEShuffleRead" not in txt_after_count, (
    "count() 之后就已经能看到 Final Plan ⇒ 本镜像的 finalize 时机与本篇正文（④a 那张顺序表）不符，"
    "**改的是正文那三行表格**，别删这条断言"
)

parts_actual = agg.rdd.getNumPartitions()   # ← 这一步才真的把 AQE 的 Final Plan 逼出来（实测）
txt_final = fmt(agg)
print("--- Final Plan（只留带 AQE / Statistics 的行）---")
print("\n".join(ln for ln in txt_final.splitlines() if "AQE" in ln or "Statistics" in ln or "== " in ln))

assert "== Final Plan ==" in txt_final and "AQEShuffleRead" in txt_final, txt_final[:400]
assert parts_actual < parts_planned, (parts_actual, parts_planned)
print(f"配置写的分区数 {parts_planned} → AQE 合并后读侧实际分区数 {parts_actual}")

m_stat = re.search(r"ShuffleQueryStage[^,]*?, Statistics\(sizeInBytes=([\d.]+) ([A-Za-z]+), rowCount=([\d.eE+]+)\)", txt_final)
assert m_stat, txt_final[:400]
UNIT = {"B": 1, "KiB": 1024, "MiB": 1024 ** 2, "GiB": 1024 ** 3}
shuffle_bytes = int(float(m_stat.group(1)) * UNIT[m_stat.group(2)])
print(f"AQE 依据的那个数就在计划里：ShuffleQueryStage 的 Statistics(sizeInBytes={m_stat.group(1)} {m_stat.group(2)} = {shuffle_bytes} 字节, rowCount={m_stat.group(3)})")
assert shuffle_bytes > 0
SHUFFLE_BYTES = shuffle_bytes   # ④c 那格拿它跟 skewJoin 阈值比
print("WI90[03-reading-the-plan-and-aqe][aqe-coalesces-shuffle-read] OK")

In [ ]:
# marker aqe-coalescing-is-one-switch gate 的是**一句**结论：
# "决定读侧要不要合并的是 coalescePartitions 那一个子开关，不是 adaptive.enabled 那个大开关"。
# 三格对照就是那句话的全部证据，assert 只有一条（三元组逐位相等），它判的是整句话。
# 第三格额外断言 ShuffleQueryStage/Statistics 两行还在：那是"大开关开着、统计照收、只是没合"的物证 ——
# 少了它，"关掉子开关"就可能被读成"AQE 什么都没做"。
# 第四格反过来断第一格 ShuffleQueryStage **一行都没有**：关掉大开关连统计采集那层都消失，
# 这一条就是 5.4 第 3 条"动大开关等于拿掉三样能力"的依据。
rows = {}
for adaptive, coalesce in ((False, True), (True, True), (True, False)):
    spark.conf.set("spark.sql.adaptive.enabled", adaptive)
    spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", coalesce)
    frame = big.groupBy("k").agg(f.sum("v").alias("s"), f.count(f.lit(1)).alias("n"))   # 每格都新建
    _ = frame.count()
    n = frame.rdd.getNumPartitions()
    txt = fmt(frame)
    rows[(adaptive, coalesce)] = (
        n,
        txt.count("AQEShuffleRead"),
        txt.count("ShuffleQueryStage"),
        len(re.findall(r"Statistics\(sizeInBytes=", txt)),
    )
    print(f"adaptive.enabled={adaptive!s:<5} coalescePartitions.enabled={coalesce!s:<5} → 读侧分区 {n:>3} | AQEShuffleRead 行 {rows[(adaptive, coalesce)][1]} | ShuffleQueryStage 行 {rows[(adaptive, coalesce)][2]} | Statistics 行 {rows[(adaptive, coalesce)][3]}")

grid = tuple(v[0] for v in (rows[(False, True)], rows[(True, True)], rows[(True, False)]))
assert grid == (200, 1, 200), (
    f"三格对照现在是 {grid}（期望 (200, 1, 200)）⇒ 『合并归那一个子开关』这句话不再成立。"
    f"逐格看：大开关关 {grid[0]} / 两格都开 {grid[1]} / 只关子开关 {grid[2]}。"
    "红在这里改的是本篇正文那三行表格与 ⑥ 速查表那一行，不是这条断言"
)
assert rows[(True, True)][1] > 0 and rows[(True, False)][1] == 0, rows
assert rows[(True, False)][2] > 0 and rows[(True, False)][3] > 0, rows
assert rows[(False, True)][2] == 0, rows
print("⇒ 只有 (开, 开) 那一格出现 AQEShuffleRead 并把 200 合成 1；")
print("   (开, 关) 那一格 ShuffleQueryStage 与 Statistics 都还在 ⇒ 统计照收，合并不做了。")
print("WI90[03-reading-the-plan-and-aqe][aqe-coalescing-is-one-switch] OK")

# 合并粒度的四档（不给 marker：它们是 ⑥ 参数表里 minPartitionSize / advisoryPartitionSizeInBytes /
# parallelismFirst 那三行的实测依据 —— 表里敢写"实测"就得在这里打印，纪律③）。
# ⚠ 每一档都把**三个旋钮一起显式设**：第一版只设被变的那个，结果每一档都继承了上一档的设置，
#    量出来的 "advisory=1b ⇒ 44 个分区" 其实是 "minPartitionSize 也还是 1b" 的合力（实测踩过，改正如下）。
def read_after_coalesce(min_size="1MB", advisory="64MB", pfirst="true"):
    spark.conf.set("spark.sql.adaptive.enabled", True)
    spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", True)
    spark.conf.set("spark.sql.adaptive.coalescePartitions.minPartitionSize", min_size)
    spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", advisory)
    spark.conf.set("spark.sql.adaptive.coalescePartitions.parallelismFirst", pfirst)
    fr = big.groupBy("k").count()      # 每档新建
    fr.count()
    return fr.rdd.getNumPartitions()


ladder = {
    "出厂三件套 (1MB / 64MB / true)": read_after_coalesce(),
    "minPartitionSize=1b": read_after_coalesce(min_size="1b"),
    "advisory=1b": read_after_coalesce(advisory="1b"),
    "parallelismFirst=false": read_after_coalesce(pfirst="false"),
}
for label, n in ladder.items():
    print(f"{label:<34} → 读侧分区 {n}")
spark.conf.set("spark.sql.adaptive.coalescePartitions.minPartitionSize", "1MB")
spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "64MB")
spark.conf.set("spark.sql.adaptive.coalescePartitions.parallelismFirst", "true")

base = ladder["出厂三件套 (1MB / 64MB / true)"]
assert base == 1, ladder                       # 与 ④a 同一个数，两格要能互相对上
assert ladder["minPartitionSize=1b"] > base, ladder
print("⇒ 把每分区下限压到 1 字节也只回到 3 个分区（不是 200）：合并目标还有一层按并行度取的逻辑。")
print("   另外两档只打印、不断言（『advisory 目标大小』与『parallelismFirst』在这份小数据上的作用范围本篇没验到底，见 ⑥ 5.5）")

## ④c 动态 join 策略：planner 与 AQE 读的是两份**不同来源、不同时刻**的大小

`bd-pyspark-0010` 那道题的追问就落在这里：**同一句等值 join，为什么"计划"和"真正跑的那份"可以不一样，
以及"AQE 会不会替你把大 join 救回来"。** 四格对照（每格都新建 DataFrame），实测：

| `adaptive.enabled` | 主阈值 | `adaptive.autoBroadcastJoinThreshold` | 初始计划 | Final 计划 |
| --- | --- | --- | --- | --- |
| 开 | `-1` | `10MB` | `SortMergeJoin` | **`BroadcastHashJoin BuildLeft`** |
| 开 | `-1` | `-1` | `SortMergeJoin` | `SortMergeJoin`（没变） |
| 开 | `10MB` | `-1` | `BroadcastHashJoin BuildRight` | 不变（**这不是 AQE 干的**，planner 一开始就选了广播 `dim`） |
| 关 | `-1` | `10MB` | `SortMergeJoin` | `SortMergeJoin`（子开关在没有大开关时**不起作用**） |

第一行是重点，也是"两个开关"最容易被讲反的地方：主阈值设 `-1`（"谁都别广播"），
但 `spark.sql.adaptive.autoBroadcastJoinThreshold` 单独放开 ⇒ AQE 在两侧 shuffle 物化之后按**真实统计**
重新决定 join 策略，把它换成了广播。⇒ 生产上"我明明把 `autoBroadcastJoinThreshold` 设成 -1 了，
计划里怎么还有 `BroadcastExchange`"就是这么来的：**那是另一个键放开的**
（会话单元实测它出厂 `<undefined>` ⇒ 继承主键；两者都显式设过时就以它为准）。

⚠ 诚实交代两件事（都是这一格的实测，别把没验的讲成验的）：

- 第一格被广播的是**左边**那一坨已经落盘的 shuffle 输出
  （`BroadcastQueryStage … Statistics(sizeInBytes=8.0 MiB, rowCount=2.00E+5)`，实测），
  不是更小的 `dim`（它的 shuffle 输出实测 `1200.0 B, rowCount=50`）。
  本篇只断到"**策略从 `SortMergeJoin` 变成了 `BroadcastHashJoin`**"，
  **为什么挑了这一侧本篇没有验**（那牵涉 `JoinSelection` 的候选排序，见 5.5）。
- ③ 那两份**编译期**估计（`dim` 600 B / `big` 3,200,000 B）与这里的**运行时**统计
  （两侧 shuffle 分别实测 `6.1 MiB` 与 `1200.0 B`，被广播那份 `8.0 MiB`）是两批不同的数、量的是不同的东西：
  前者挂在 logical plan 的 `stats()` 上，planner 生成初始计划时用掉就丢；
  后者由 shuffle writer 落完之后汇总成 `MapOutputStatistics`，挂在 `ShuffleQueryStage` 节点上。
  ⇒ 这一格会把它断成数字：**执行之前 `explain("formatted")` 的文本里 `Statistics(` 出现 0 次**（实测），
  物化并 finalize 之后才出现。这就是 5.4 第 3 条"AQE 为什么看不到 CBO"的证据本体。
  （本镜像 `spark.sql.cbo.enabled` 实测出厂 `false` ⇒ 那份编译期估计连表统计都不是，只是默认行宽：
  `big` 的 3,200,000 B = 20 万行 × 每行 16 B；本篇只观察到这个乘积，默认行宽那条规则本身没验。）

⚠ skewJoin 那一段（三段能力里的第三段）在本机**没有任何可断言的行为**：
下面那一格会把 ④a 量到的 shuffle 总字节与出厂阈值 `256MB`（会话单元打印的那个键）摆在一起比 ——
④a 量到的 shuffle 总量 3,174 B ≪ 256MB ⇒ 任何一个分区都不可能过阈值，"AQE 把倾斜分区劈开了"这件事在本镜像**不可能被观察到**。
所以本篇不断言它生效，只断言"它此刻不可能生效"（这条是确定量，坏了走执行层）。
真实集群上它的触发条件、`forceOptimizeSkewedJoin` 的代价、以及"为什么聚合不归它管"都是篇 1 的口径，本篇不复述。

In [ ]:
# 这一格**不打 marker**（四条结论已经各归各的格子，再加一条就是两个下标挂同一件事）：
# 它判的是"动态 join 策略四格对照"，坏了走闸门的执行层，红话说的是"本节正文与 ⑥ 那张表过期了"。
spark.conf.set("spark.sql.shuffle.partitions", 200)
arms = {}
stats_before_all, stats_after_all = [], []
for adaptive, base, adapt in (
    (True, -1, 10 * 1024 * 1024),
    (True, -1, -1),
    (True, 10 * 1024 * 1024, -1),
    (False, -1, 10 * 1024 * 1024),
):
    spark.conf.set("spark.sql.adaptive.enabled", adaptive)
    spark.conf.set("spark.sql.autoBroadcastJoinThreshold", base)
    spark.conf.set("spark.sql.adaptive.autoBroadcastJoinThreshold", adapt)
    j = big.join(dim, "k", "inner")                 # 每格新建
    initial = simple(j)
    n_stat_before = fmt(j).count("Statistics(")     # 执行之前：运行时统计那几行还不存在（实测 0）
    j.count()
    j.rdd.getNumPartitions()                        # 与 ④a 同一招：读一次 rdd 才 finalize
    final_txt = fmt(j)
    head = final_txt.split("== Final Plan ==")[-1]
    if "== Initial Plan ==" in head:                # 只看 Final Plan 那一段，别把 Initial 里的算子当成结果
        head = head.split("== Initial Plan ==")[0]
    op_i = re.search(r"(SortMergeJoin|BroadcastHashJoin)", initial)
    op_f = re.search(r"(SortMergeJoin|BroadcastHashJoin)", head)
    arms[(adaptive, base, adapt)] = (op_i.group(1), op_f.group(1))
    stats_before_all.append(n_stat_before)
    stats_after_all.append(final_txt.count("Statistics("))
    stats = [ln.strip() for ln in final_txt.splitlines() if "Statistics(sizeInBytes" in ln]
    print(f"adaptive={adaptive!s:<5} 主阈值={base:>10} 自适应阈值={adapt:>10} → 初始 {op_i.group(1):<17} Final {op_f.group(1):<17} | Statistics( 次数 执行前 {n_stat_before} / 执行后 {final_txt.count('Statistics(')}")
    print("      运行时统计（执行前的文本里一条都没有）:", stats)

assert arms[(True, -1, 10 * 1024 * 1024)] == ("SortMergeJoin", "BroadcastHashJoin"), arms
assert arms[(True, -1, -1)] == ("SortMergeJoin", "SortMergeJoin"), arms
assert arms[(True, 10 * 1024 * 1024, -1)][0] == "BroadcastHashJoin", arms
assert arms[(False, -1, 10 * 1024 * 1024)] == ("SortMergeJoin", "SortMergeJoin"), arms
# "两套大小不同源"的判据：AQE 那一侧的 Statistics( 只能由执行产出（执行前 0 次）；大开关关掉则这一层彻底不存在。
assert stats_before_all == [0, 0, 0, 0], stats_before_all
assert all(x > 0 for x in stats_after_all[:3]), stats_after_all
assert stats_after_all[3] == 0, stats_after_all
print("⇒ 只有第一格是 AQE 改写的：主阈值关着、自适应阈值放开；大开关一关，自适应阈值就完全不起作用（第四格）。")

# skewJoin：本机不可能触发的确定量（断的是"不可能"，不是"生效"）
skew_bytes = int(spark.conf.get("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes").rstrip("b"))
print(f"shuffle 总量 {SHUFFLE_BYTES} 字节 vs skewJoin 阈值 {skew_bytes} 字节 ⇒ 任何一个分区都不可能过阈值")
assert SHUFFLE_BYTES < skew_bytes, (SHUFFLE_BYTES, skew_bytes)
print("⇒ 所以本篇不断言 skew join 生效（那是一条永远随机红的断言）；真实集群上的触发条件见 ⑥ 与篇 1 解法 C")

spark.conf.set("spark.sql.adaptive.autoBroadcastJoinThreshold", -1)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10 * 1024 * 1024)

## ⑤ 耗时对照：**只打印两个数**，"更少分区 ≠ 更快"这句话就停在这里

下面那一格跑 6 次同一句聚合（AQE 开/关交错三轮），打印每次的墙上时间与读侧分区数。
它**没有任何 `assert`**，也不是任何一个 marker 的所在地。

为什么这里一条断言都不加，而这一篇别处断言加得很凶？因为**"更快/更慢"在本机不是确定量**，
而一条押耗时的 `assert` 就等于把闸门交给机器负载：判题池抢 CPU 时它先红，
而红出来的话是"教程坏了"。（同一条纪律在篇 1 的 ④、篇 2 的 ④ 各写过一次，三篇一致。）

本机实际打出来的方向是：**AQE 开那一侧更快**（读侧 1 个分区 vs 200 个）。
但这**不构成"合并就是更快"的结论**，两条理由都要写下来：

1. 那 200 个分区里只有 44 个非空（② 实测），本机又没有网络 shuffle ⇒
   "200 个 task"的代价在这里只是**调度 156 个空 task** 的费用，跟真实集群上"并行度被合没了"完全不是一回事。
2. 真实集群上过度合并的失效模式是"下游只剩几个 task、一个跑 40 分钟"——
   那是节点级现象，`local[2]` 一个 JVM 两个线程，**造不出来（未验证）**。

所以"更少分区 ≠ 更快"在本篇的地位是**告诫 + 未验证**，不是实测结论：
它教的是"别把 AQE 的合并当免费"，而它的**反例方向**要你在真集群上用 task 时长分布去拿
（线上就在 Spark UI 那个 stage 上看每个 task 的 *Duration* 与 *Shuffled Read Size* 分布；查哪三个数见 ⑥ 的速查表。
 本篇没有那道题的现场，也不假装引用哪道题 —— 引错的题号比不引更贵）。
本篇能负责任地说出口的只有确定量：**合并发生了，读侧从 200 变成 1，统计来源是 `ShuffleQueryStage` 那行。**

In [ ]:
# 这一格只打印。它是导语里那两个秒数的现场，也是"更少分区 ≠ 更快"那句话的**唯一**数字依据 ——
# 而那句话按纪律写在 ⑤ 的 markdown 里，不进这里任何一条 assert（这一格一条 assert 都没有）。
def timed_agg(label, adaptive):
    spark.conf.set("spark.sql.adaptive.enabled", adaptive)
    spark.conf.set("spark.sql.shuffle.partitions", 200)
    frame = big.groupBy("k").agg(f.sum("v").alias("s"), f.count(f.lit(1)).alias("n"))   # 每次都新建
    t = time.time()
    n_rows = frame.count()
    secs = round(time.time() - t, 3)
    n_read = frame.rdd.getNumPartitions()
    print(f"{label:<12} count() = {secs:>6} s | 读侧分区 {n_read:>3} | 行数 {n_rows}")
    return secs, n_read


print("三轮交错（每轮 开→关 各一次），只为让人看见数字本身的抖动：")
samples = {"AQE 开": [], "AQE 关": []}
for rnd in range(3):
    for label, adaptive in (("AQE 开", True), ("AQE 关", False)):
        samples[label].append(timed_agg(f"{label} 第{rnd + 1}轮", adaptive))
print()
print("读侧分区：AQE 开 =", [s[1] for s in samples["AQE 开"]], "| AQE 关 =", [s[1] for s in samples["AQE 关"]], "← 这一行是确定量，但它已由 ④a/④b 判过")
print("耗时：AQE 开 =", [s[0] for s in samples["AQE 开"]], "| AQE 关 =", [s[0] for s in samples["AQE 关"]], "← 只打印，不作结论")
print("⇒ 本机这三对数字的方向（开更快）来自『200 个桶里 156 个是空的、又没有网络 shuffle』，")
print("   它既没证明也没证伪『更少分区 ≠ 更快』；那句话的形态只在真实集群的 task 时长分布上才成立。")
spark.conf.set("spark.sql.adaptive.enabled", True)

## ⑥ 速查表（可以只靠它回答问题）+ 追问链

### 5.1 拿到一份物理计划，先读这三行（本篇的主问题）

| 顺序 | 读哪一行 | 读出来的东西 | 读完你就知道下一步查什么 |
| --- | --- | --- | --- |
| 1 | `Range(..., splits=N)` / `FileScan parquet ... ReadSchema` / `Exchange hashpartitioning(键, N)` | **并行度从哪来**：源头 N 个分区 + 每次 shuffle N 个桶 | N 只有 2 ⇒ 慢在 map 端就要查源头分区数（读侧装箱是篇 2）；N=200 而键基数只有 50 ⇒ 有 150+ 个空桶（② 实测 44），"加分区"不是解法 |
| 2 | 那个 `Exchange` **下游的第一个算子**：`HashAggregate` 还是 `SortMergeJoin` / `BroadcastHashJoin` | **倾斜该找谁**：聚合的倾斜只有加盐（篇 1），join 的倾斜才有广播/AQE skew join | `SortMergeJoin` + 两侧 Exchange ⇒ join 的 shuffle 是瓶颈，先问"能不能广播、小表真的还小吗"（③、`bd-pyspark-0002`） |
| 3 | `*(N)` 的分组，和 `AdaptiveSparkPlan isFinalPlan=?` / `AQEShuffleRead` / `ShuffleQueryStage, Statistics(...)` 这几行有没有出现 | **你读的这份计划是不是真跑的那份**：`*(N)` 是 codegen stage 编号不是分区数；有 `AQEShuffleRead` ⇒ AQE 已经改写过，你看到的桶数不是配置里那个 | 没执行之前只有 Initial Plan ⇒ 别拿它断言 AQE 的行为（④a 实测那个文本里 `AQEShuffleRead` 出现 0 次） |

### 5.2 "我线上 shuffle 后有个 task 跑了 40 分钟"—— 只靠这张表怎么答（本篇的口径：先问计划，再问分布）

三个数，全都不依赖时间：

1. **那个 stage 对应的 `Exchange hashpartitioning(键, N)` 里的 N，与键的实际基数**。
   怎么查：把同一句查询的 `simple()` 打出来（② 那格）；基数用 `df.select(键).distinct().count()`。
   N ≫ 基数 ⇒ 并行度是配置写的、多出来的桶是空的（② 实测 200 个桶只落 44 个非空文件），
   "40 分钟那个 task"多半不是并行度问题，而是**键分布**问题 ⇒ 走第 3 个数。
2. **那个 Exchange 的下游算子是谁**（`HashAggregate` 还是 join）。
   这决定了两条完全不同的验证路径（见 5.1 第 2 行）。
3. **那一次 shuffle 的真实字节/行数**（`ShuffleQueryStage, Statistics(sizeInBytes=…, rowCount=…)`，④a/④c 实测这一行只有执行之后才有；
   线上就在 Spark UI 那个 stage 的 *Shuffled Read Size / Records*）。
   拿它跟 `skewedPartitionFactor × median` 和 `skewedPartitionThresholdInBytes`（本镜像实测 5.0 / 256MB）比 ⇒
   过不了阈值 ⇒ **AQE 的 skew join 一定不会动**，别再说"我 AQE 开了呀"。

**最可能的两种根因，以及各自怎么验证**：

| 根因 | 计划上的痕迹（本篇教的读法） | 验证动作 | 解法归哪篇 |
| --- | --- | --- | --- |
| **键倾斜**（一个 key 吃掉大半行） | `Exchange hashpartitioning(键, N)` 的 N 正常，但下游只有一个 task 慢 | 数每个桶的行数占比（`groupBy(键).count()` 后按桶取 max/total，或 UI 上按 *Task Descriptor* 的 records 排序）；再用 `hashpartitioning` 那一行的键确认"我量的那次 shuffle 就是查询里那次" | 篇 1：聚合 → 加盐两阶段；join → 广播 / 拆热点 key / skew join（要过阈值） |
| **并行度不对**（桶数 ≫ 键基数，或 AQE 把读侧合得太狠） | ① `hashpartitioning(键, 200)` 而键基数远小于 200（空桶）；② 计划里有 `AQEShuffleRead` 而读侧只剩 1 个分区（④a 实测） | 执行前取计划确认"配置写的那个数"，执行后取 Final Plan 与 `df.rdd.getNumPartitions()` 确认"AQE 改出来的那个数"，两个都要记（本篇 ②/④a 就是这两次取值） | 本篇 ④b：调 `coalescePartitions.enabled` / `minPartitionSize`，**不是**关 `adaptive.enabled` 那个大开关 |

### 5.3 参数表（"本镜像生效值"那一列全是会话单元打印的，不是我背的）

| 参数 | 本镜像实测 | 什么时候动 | 动错的表征 |
| --- | --- | --- | --- |
| `spark.sql.shuffle.partitions` | `200`（② 当场把它改 8 又改回来） | 只决定"shuffle 之后开多少个桶" | 调太大 ⇒ 空桶与碎 task（②：50 个键 @200 桶只落 44 个非空文件）；调太小 ⇒ 单 task 太大、spill |
| `spark.sql.adaptive.enabled` | 出厂 `true`（会话单元断成前提） | 度量/教学时关；生产开 | **它不是"合并"的开关**：一关，`ShuffleQueryStage`/统计/三种改写全没了（④b 第一格 200）；开着时你设的 200 只是上限 |
| `spark.sql.adaptive.coalescePartitions.enabled` | `true` | 读侧并行度被合没了（④b 第三格就是它的效果演示） | 关掉 ⇒ 200 个分区照旧开 200 个 task（空桶也一样开）；开着 ⇒ 本例合到 1 |
| `spark.sql.adaptive.coalescePartitions.minPartitionSize` | `1MB` | 给"合并后每个分区"托个底 | 实测下限压到 `1b` 也只回到 3 个分区（不是 200）：还有一层按并行度取的逻辑，本篇没验到底 |
| `spark.sql.adaptive.coalescePartitions.parallelismFirst` | `true` | 想让合并只看目标大小、不被并行度牵着走 | ④b 把它单独变档打印了（其余两键回出厂），**但没断言**：这份数据只有 3 KiB，两个方向都看不出差别 ⇒ 别拿本篇当"它没用"的依据 |
| `spark.sql.adaptive.advisoryPartitionSizeInBytes` | `conf.get` = `67108864b`（64MB），`SET -v` 里那一行显示成 `<value of spark.sql.adaptive.shuffle.targetPostShuffleInputSize>`（别名） | join/shuffle 侧目标分区大小 | 与 `minPartitionSize` 混为一谈 ⇒ "为什么合并动不动"。④b 把它单独压到 `1b` 打印（其余两键回出厂），**没断言**；本篇的读侧判据用的是 `df.rdd.getNumPartitions()` 而不是它 |
| `spark.sql.adaptive.coalescePartitions.initialPartitionNum` | `<undefined>`（`conf.get` 给 Python `None`） | 想让 AQE 从一个更大的初值往下合 | 与 `shuffle.partitions` 搞混 ⇒ "我设了 200 为什么 AQE 看到的是别的数" |
| `spark.sql.autoBroadcastJoinThreshold` | `10485760b`（10MB；`SET -v` 写 `10MB`） | 强制/禁止自动广播（③ 两头都调过） | 设太大 ⇒ build 侧撑爆内存（`bd-pyspark-0002` 的 18GB 维度表就是这一类）；`-1` 关自动广播 |
| `spark.sql.adaptive.autoBroadcastJoinThreshold` | **`<undefined>` ⇒ 继承主键**（会话单元那行 `conf.get` 给 `None`，实测） | 想"planner 别广播，但 AQE 按真实统计可以广播"（④c 第一格） | 以为主阈值设 `-1` 就绝对不会广播 ⇒ 结果线上出现 `BroadcastExchange`：那是这个子开关放开的，④c 当场复现了 |
| `spark.sql.adaptive.skewJoin.enabled` / `skewedPartitionFactor` / `skewedPartitionThresholdInBytes` | `true` / `5.0` / `268435456b`（256MB） | 真集群上 join 倾斜且分区够大 | 以为它管聚合（篇 1 最主要的那条误解）；本机 shuffle 总量实测 3,174 B ≪ 256MB ⇒ **本篇一次都没触发过它** |
| `spark.sql.adaptive.localShuffleReader.enabled` | `true` | 跨节点时省掉重复读 | `local[2]` 没有跨节点 ⇒ 本机量不到它的价值（未验证） |
| `spark.sql.cbo.enabled` | `false` | 想让 planner 用表统计选 join 顺序/策略 | 开着却没 `ANALYZE TABLE` ⇒ 估计照旧是默认行宽（③ 那份 600 B / 3,200,000 B 就是没有表统计时的估计） |
| `defaultParallelism`（不是 Spark 配置，是 `local[2]` 给的） | `2` | 本机所有"并行度"数字的分母 | 看到"200 个桶合成 1 个分区""读 2 个 parquet 只有 2 个输入分区"别慌，都以 2 为分母 |

**三句就够的形状（背这三句）**：
`计划里那个 shuffle 数 = 配置写的`；`AQE 改出来那个数 = 运行时统计写的`；
`要管合并就动 coalescePartitions，不要动 adaptive.enabled 那个大开关`。

### 5.4 面试官追问链（第 3 条是机制层，本篇的落点）

1. **"这份 plan 你先读哪三行？为什么不是先读最慢的那个算子？"**
   先读 `Exchange hashpartitioning(键, N)`（并行度是谁写的）、它**下游的第一个算子**（决定倾斜归聚合还是归 join）、
   源头那行（`splits` / `FileScan`）。最慢的算子要等执行完才有，而计划在你按下执行前就能否掉一半假设 ——
   例如 `N=200` 而键基数只有 50，这个查询**不可能**靠再加并行度变快（② 实测 156 个空桶）。
   追问：**"`*(2)` 是不是两个分区？"** —— 不是，`*(N)` 是 WholeStageCodegen 的 stage 编号（① 实测：两个 `HashAggregate` 分别在 `*(1)`/`*(2)`，中间隔着一条 Exchange）。这是最常被抓的读法错。
2. **"你说 AQE 把 200 合成了 1，那我把 `shuffle.partitions` 设成 2000 是不是就更有并行度？"**
   两个数都不是"并行度"：`shuffle.partitions` 是**上限**（④b：开着 AQE 时你设 200 拿到 1），
   键基数才是**上限的上限**（②：50 个键 ⇒ 44 个非空）。要真提高并行度得动**键的分布**（篇 1 的加盐）
   或把合并关掉/下限压低（④b 的 `minPartitionSize`，实测 1b ⇒ 3 个分区）。
   追问：**"那 `coalesce(8)` 呢？"** —— `coalesce` 只能降不能升（篇 2 实测 2 分区的源 `coalesce(8)` 还是 2），
   而且它不打 Exchange，所以计划上看得见的是 `Coalesce 8` 而不是 `RoundRobinPartitioning(8)`（篇 2 ③ 的判据）。
3. **"AQE 为什么看不到 CBO？"**（机制层，本篇的主回答）
   两套统计**住在不同的地方、在不同时刻被读**。CBO 的那份挂在 logical plan 的 `stats()` 上
   （`optimizedPlan().stats().sizeInBytes()`，本篇实测 `dim`=600 B、`big`=3,200,000 B、join 输出估计 1,600,000,000 B），
   它是 **planner 在生成初始物理计划之前**用的；本镜像 `spark.sql.cbo.enabled` 实测出厂 `false`，
   连这份估计都只是默认行宽，不是表统计。AQE 的那份是 shuffle writer 落完之后由统计跟踪器汇总的
   `MapOutputStatistics`，物理上挂在 `ShuffleQueryStage` 节点上 —— 所以
   `explain("formatted")` 在**执行之前连 `Statistics(` 这一行都没有**（④c 实测），执行之后才有。
   AQE 的回调发生在物理执行的中途：那一刻 logical 层的活（列裁剪、谓词下推、join 顺序）已经定完，
   它手里唯一的新信息就是刚物化那个 stage 的字节/行数，于是它能做的只有
   **合分区 / 重挑 join 策略与构建侧 / 劈倾斜分区**这三件事。
   ⇒ 推论（能直接回答"要不要开 AQE 就不用调优了"）：**AQE 不会替你补 CBO 的课**：
   join 顺序错了、缺列统计导致的选择错误，AQE 看不见也改不了；反过来 `ANALYZE TABLE` 也帮不了
   AQE 决定"要不要把 200 个 3KiB 的分区合成 1 个"。两件事各自要付成本。
   追问：**"为什么合并读侧的是 `coalescePartitions` 那一个开关，不是 AQE 这个大开关？"**
   大开关决定"走不走 AdaptiveSparkPlanExec 这条路径"（关掉 ⇒ 统计采集与三条规则全消失，④b 第一格 200、
   且 `ShuffleQueryStage` 一行都没有）；"要不要把已经量出来的小分区合掉"是这条路径里**一条叫
   CoalesceShufflePartitions 的规则**，有独立开关（④b 第三格：200 照旧，但 `ShuffleQueryStage` 与
   `Statistics(…)` 两行**都还在** ⇒ 统计照收，只是不合并）。所以线上"合并把并行度压没了"的处置是动子开关，
   把大开关关掉会连动态 join 策略与 skew join 一起失去 —— 那是拿掉三样能力去换一样。
4. **"怎么证明 broadcast 把 executor 撑爆了？"**
   诚实答案：**本机证不了**（kernel 是 `local[2]`：一个 JVM 两个线程，没有"某个 executor 独立内存被打爆"这条线）。
   能当场看的只有两样：计划里被广播那一侧的**规模**（编译期 `stats().sizeInBytes()`，③/④c 都打印了；
   运行时 `BroadcastQueryStage, Statistics(sizeInBytes=8.0 MiB)`，④c 打印了）
   与 `spark.sql.broadcastTimeout`（篇 1 的会话单元打印过它的出厂值）。
   线上要拿的是 executor 级指标：build 阶段的 GC time / spill、`BroadcastExchange` 那个 stage 的 task 时长、
   driver 侧 `Blocked broadcast exchange build` 日志与 OOM 退出记录 —— **本篇未验证**。
   `bd-pyspark-0002` 就是这条的现场（18GB 维度表 + 多版本膨胀：广播前必须先把维度收敛到"一行一键"）。
5. **"非等值 join 为什么计划会退化？"**
   本篇四格对照里能看到的都是**等值** join：`k` 同时是两侧 `hashpartitioning` 的分区键 ⇒ 才有
   `SortMergeJoin`/`BroadcastHashJoin` 这两种选择。条件变成 `a.ts between b.from and b.to` 这种非等值时，
   没有任何键能把两侧对齐到同一个分区 ⇒ hash 策略全部不可用，只剩 nested-loop 那一族
   （小表侧就是 `BroadcastNestedLoopJoin`，两边都大就是 `笛卡尔 + filter`，那是"几十亿行"的来源）。
   解是把非等值**改写成等值 + 过滤**（按 `user_id` 等值 join 再在窗口内过滤），这正是 `bd-pyspark-0010` 的题面。
   本机没有跑那个改写（**未验证**：本篇只量了等值那一支的计划形状）。

### 5.5 本篇没验的东西（照纪律逐条列出）

- **skew join 的任何实际行为**：④c 断的是"本机不可能触发"（shuffle 总量 3,174 B ≪ 256MB），不是"它会怎么改"。
  `OptimizeSkewedJoin` 这个算子名在本篇所有计划里**一次都没出现过**。
- **节点级失效模式**：`FetchFailedException`、executor OOM、长尾 task 拖垮节点、broadcast 撑爆某个 executor
  —— `local[2]` 一个 JVM 两个线程，产生不了。`spark.sql.adaptive.localShuffleReader` 的价值同理（没有跨节点）。
- **"更少分区 ≠ 更快"的方向**：⑤ 那三对本机是 AQE 一侧更快（0.05s 级 vs 0.41~0.47s 级，秒数逐轮漂），
  既没证明也没证伪那句话；它在本篇只是告诫。真实结论要 task 时长分布 + spill。
- **合并粒度的公式**：`parallelismFirst` / `minParallelismFactor` / `advisoryPartitionSizeInBytes` 三者怎么共同决定
  "合成几个分区"，本篇只量了四档（④b 打印：出厂三件套、`minPartitionSize=1b`、`advisory=1b`、`parallelismFirst=false`），
  且只有前两档进了断言；没有参数扫描，也没有大数据下的对照。
- **CBO 真开起来的效果**：`cbo.enabled=true` + `ANALYZE TABLE` 之后 planner 的估计会不会变准、
  ④c 那个反差会不会消失 —— 本篇没跑（本镜像没收集列统计）。
- **非等值 join 的退化形状**：只讲了口径（5.4 第 5 条），没执行过 `between` 条件的 join。
- **跨版本史实**：AQE 规则名（`CoalesceShufflePartitions` / `OptimizeSkewedJoin`）、各键默认值、
  `advisoryPartitionSizeInBytes` 的别名关系 —— 本镜像只有 Spark 3.5.5，**没有逐版本核实，迁移以官方配置页为准**。
- **`ReusedExchange` / `ReuseAdaptiveSubquery`**：本篇的计划里没有出现过，没验。

## 题库指针（同一套口径，不是"再找篇资料"）

- **`bd-pyspark-0002`** ——「join 右表多版本导致行数膨胀：先收敛维度再广播（结果行数必须等于左表）」（senior，判题 `pyspark`）。
  对应本篇 **③ 那一节**（阈值改的是算子名 + 两条 Exchange 的有无，以及"阈值比的是编译期规模估计"），
  也对应 **④c**（同一个 join 可以被 AQE 按真实统计改写成广播）。
  它卡的是"广播的**资格**"而不是"广播的**方向**"：维度表带多版本时行数先膨胀，
  `sizeInBytes` 越过阈值就不再自动广播；题面还写"`autoBroadcastJoinThreshold` 默认 8 MB"，
  本镜像实测出厂 **10 MB**（③ 那节末尾那条订正）——**跨版本默认值本篇没核实**，别把 10MB 当常数背。
  它的正确顺序是"先 `groupBy` 收敛到一行一键，再广播"；顺序反了就是 OOM。
- **`bd-pyspark-0010`** ——「Last-touch 归因：非等值时间窗 join + 窗口取最新（24 小时含边界的口径钉死）」（senior，判题 `pyspark`）。
  对应本篇 **① 那张地图的第 2 行**与 **5.4 第 5 条**：`Exchange hashpartitioning(键, N)` 里的"键"必须存在，
  `SortMergeJoin` / `BroadcastHashJoin` 这两条路才走得了；条件换成
  `touch_ts between win_start and win_end` 这种非等值 ⇒ 两侧无法按同一个键分区 ⇒ 退化到
  `BroadcastNestedLoopJoin` / 笛卡尔 + filter（题面原话：`touchpoints` 只有几行时会走 BroadcastNestedLoopJoin，
  线上 19 亿行必须改写成"按 `user_id` 等值 join + 窗口过滤 + `ROW_NUMBER`"）。
  ⇒ 读计划的用处就在这：**先问"这次 Exchange 的分区键是谁"**，键不存在就说明你已经不在 hash join 那一族里了。
  本篇**没有执行过**那条非等值改写（5.5 已列未验证），去那道题里量。

系列内：篇 1 = 倾斜与热点 key（键分布的量法、加盐、`hint("skew")` 在本镜像不被识别、`forceOptimizeSkewedJoin` 的方向）；
篇 2 = 小文件与物理设计（读侧装箱 `maxPartitionBytes`/`openCostInBytes`、`coalesce` 与 `repartition` 的算子名差别）。
本篇刻意**不 `cache()`**：一 cache，后面几次"换配置取计划"量的就不是同一份物理执行了；
也刻意**不显式 `repartition(N)`**：AQE 的合并不动显式 repartition（篇 2 实测），
那一格要测的恰恰是"由 `shuffle.partitions` 推导出来的 hash shuffle"。

In [ ]:
# 收尾断言留在数据单元之后、spark.stop() 之前：闸门每跑一次这一篇，都要在容器 /tmp 里少留一份 parquet。
shutil.rmtree(SCRATCH, ignore_errors=True)
assert not os.path.exists(SCRATCH), (
    f"scratch 没清干净：{SCRATCH} 还在 ⇒ 闸门每跑一次就在容器 /tmp 里留一份 big 的 parquet。"
    "红在这里说的是上面那行 rmtree 被删了/失败了，不是『多点垃圾无所谓』"
)
leftover = [d for d in os.listdir(tempfile.gettempdir()) if d.startswith("wi90-03-")]
assert not leftover, leftover

# 本篇动过的键逐个恢复。④c 显式设过 adaptive.autoBroadcastJoinThreshold，它出厂是 <undefined>（没设过），
# 设回 -1 与"没设过"不是一回事（-1 = 明确禁止；没设过 = 继承主键），所以要真正 unset。
for key in (
    "spark.sql.adaptive.enabled",
    "spark.sql.adaptive.coalescePartitions.enabled",
    "spark.sql.adaptive.coalescePartitions.minPartitionSize",
    "spark.sql.adaptive.coalescePartitions.parallelismFirst",
    "spark.sql.adaptive.advisoryPartitionSizeInBytes",
    "spark.sql.adaptive.autoBroadcastJoinThreshold",
    "spark.sql.autoBroadcastJoinThreshold",
    "spark.sql.shuffle.partitions",
):
    spark._jsparkSession.conf().unset(key)

back = {k: spark.conf.get(k) for k in (
    "spark.sql.adaptive.enabled",
    "spark.sql.adaptive.coalescePartitions.enabled",
    "spark.sql.shuffle.partitions",
    "spark.sql.autoBroadcastJoinThreshold",
    "spark.sql.adaptive.autoBroadcastJoinThreshold",
)}
print("unset 之后回到出厂：", back)
assert back["spark.sql.adaptive.enabled"] == "true", back
assert back["spark.sql.adaptive.coalescePartitions.enabled"] == "true", back
assert back["spark.sql.shuffle.partitions"] == "200", back
assert back["spark.sql.autoBroadcastJoinThreshold"] == "10485760b", back
assert back["spark.sql.adaptive.autoBroadcastJoinThreshold"] is None, (
    f"adaptive.autoBroadcastJoinThreshold unset 之后不是 None（现在是 {back['spark.sql.adaptive.autoBroadcastJoinThreshold']!r}）"
    " ⇒ 它出厂就不是『没设过』那一档，④c 那张表里『继承主键』那句要按这里读到的值改写，别删这条断言"
)
print("清理完成 | /tmp 里以 wi90-03- 开头的残留：", leftover)

In [ ]:
# 最后一个单元必须是它：闸门（tutorials.test.ts）跑完这一篇就结束，
# 不往容器里丢活着的 Spark 内核（tutorials.test.ts 常驻那组的结构性那条判的就是它，宿主档 45s 就能红）。
spark.stop()
print("stopped")